In [60]:
import torch
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification, BertForSequenceClassification, DataCollatorWithPadding

import math
import numpy as np
import matplotlib.pyplot as plt
import evaluate
import pandas as pd
import scipy


In [96]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [97]:
import pandas as pd

# Define input and output file names
input_csv = "numeric_label.csv"
train_csv = "train.csv"

# Read the CSV file
df = pd.read_csv(input_csv)

# Ensure column names are correct
df.columns = ["id", "query", "label"]

# Drop the 'id' column (not needed for training)
df = df.drop(columns=["id"])

# Save entire dataset as train.csv (no split)
df.to_csv(train_csv, index=False)

print(f"Dataset saved as {train_csv} ({len(df)} samples)")


Dataset saved as train.csv (42 samples)


In [98]:
# Load the CSV files into pandas DataFrames
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('train.csv')
test_df = pd.read_csv('train.csv')

In [99]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [100]:
print(len(train_dataset))
print(len(test_dataset))
print(len(test_dataset))

42
42
42


In [114]:
train_dataset

Dataset({
    features: ['query', 'label'],
    num_rows: 42
})

# Model parameters start 

In [115]:
learning_rate = 4.00e-05
warmup_proportion = 0.1
train_batch_size = 16
eval_batch_size = 16
num_train_epochs = 2
gradient_accumulation_steps = 1

huggingface_modelname = "bert-base-uncased"

define tokeniser

In [116]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
tokenizer = AutoTokenizer.from_pretrained(huggingface_modelname)
def tokenize_function(examples):
    return tokenizer(examples["query"],truncation=True)
#tokenize datasets 
train_datasets = train_dataset.map(tokenize_function,batched = True)
val_datasets = val_dataset.map(tokenize_function,batched = True)
test_datasets = test_dataset.map(tokenize_function,batched = True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) 

Map: 100%|██████████| 42/42 [00:00<00:00, 13540.41 examples/s]


In [117]:
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification, BertForSequenceClassification, DataCollatorWithPadding
model = BertForSequenceClassification.from_pretrained(huggingface_modelname,num_labels = 10)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [118]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [119]:
training_args = TrainingArguments(
    output_dir="test_trainer", #model checkpoints, logs, and other training outputs will be saved. ex-pytorch_model.bin
    learning_rate=learning_rate,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    weight_decay=0.01,  #L2 regularization coefficient applied to prevent overfitting by penalizing large weights.
    evaluation_strategy="epoch", #Defines how often evaluation is performed
    save_strategy="epoch", #pecifies how often model checkpoints are saved
    gradient_accumulation_steps=gradient_accumulation_steps,
    load_best_model_at_end=True,
    )

/home/xrspace/.cache/pypoetry/virtualenvs/llama-xut4REbk-py3.12/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [120]:
import numpy as np
accuracy_metric  = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
#The structure of eval_pred is usually a tuple: (predictions, labels).
#eval_pred argument is supplied by the Trainer and contains 1- Predictions: Model output predictions (usually logits or probability scores).2- Labels: Ground truth labels from the dataset.
def compute_metrics(eval_pred): 
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
        # Calculate accuracy
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    
    # Calculate F1 score (macro or weighted based on your use case)
    #f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    
    # Combine metrics into a single dictionary
    metrics = {
        "accuracy": accuracy["accuracy"],
        #"f1": f1["f1"],
    }
    return metrics

In [121]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_datasets.shuffle(seed=42),
    eval_dataset=val_datasets.shuffle(seed=42),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_5226/2317664009.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [13]:
trainer.train()

  0%|          | 0/15 [00:00<?, ?it/s]../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
